<a href="https://colab.research.google.com/github/sethkipsangmutuba/Database-Management-System/blob/main/f6.%20Relational_Database_Design_by_ER_and_EER_to_Relational_Mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Section 9: Relational Database Design by ER- and EER-to-Relational Mapping

---

### Overview

This section addresses how to transform a conceptual schema (ER/EER model) into a **relational database schema**, a key step in **logical database design**.

---

### Objective

- **Bridge the gap** between:
  - Conceptual design (ER/EER models)
  - Logical implementation (Relational model)

---

### Relevance

- Complements **database design phases** outlined in *Figure 7.1*.
- Emphasizes practical **conversion of design diagrams into relational schemas** used in real DBMSs.

---

### Tool Support

- Many **CASE tools** are built on ER or EER models.
- These tools often:
  - Allow **graphical schema design**
  - **Auto-generate DDL** (Data Definition Language) using algorithms like those presented here.

---

### Chapter Structure

#### **Section 9.1**
- Details a **seven-step algorithm** to map basic ER constructs into a relational schema:
  - **Strong/weak entity types**
  - **Binary relationship types** (1:1, 1:N, M:N)
  - **n-ary relationships**
  - **Simple, composite, and multivalued attributes**

#### **Section 9.2**
- Extends the mapping to **EER constructs** such as:
  - **Specialization/Generalization**
  - **Union types (Categories)**

#### **Section 9.3**
- Provides a **concise summary** of:
  - The mapping procedure
  - Key takeaways

---

### Importance

- Enables **accurate transformation** from high-level data modeling to real-world database implementation.
- Ensures **semantic consistency** between **design** and **deployment**.


## 9.1 ER-to-Relational Mapping Algorithm

This section describes a **seven-step process** to map an ER schema to a relational database schema.

---

### **Step 1: Mapping of Regular (Strong) Entity Types**

- For each **strong entity type**, create a **relation**.
- Include **all simple attributes**.
- For **composite attributes**, include only the **simple components**.
- Choose the **primary key** of the entity type as the **primary key** of the relation.

---

### **Step 2: Mapping of Weak Entity Types**

- Create a **relation** for each **weak entity type**.
- Include **all simple attributes**.
- Include the **primary key of the identifying (owner) entity**.
- The **primary key** of the relation is the **combination of the owner's key and the weak entity's partial key**.
- Include any **relationship attributes** in the relation.

---

### **Step 3: Mapping of Binary 1:1 Relationship Types**

- Choose one of the **participating entity types**.
- Add the **primary key of the other** as a **foreign key**.
- Include any **attributes of the relationship**.
- Apply **total participation rule**:
  - Foreign key must be **NOT NULL** if **total** participation exists.

---

### **Step 4: Mapping of Binary 1:N Relationship Types**

- Identify the entity type on the **N-side**.
- Include the **primary key of the 1-side** as a **foreign key** in the N-side relation.
- Add any **attributes of the relationship**.

---

### **Step 5: Mapping of Binary M:N Relationship Types**

- Create a **new relation** for the relationship.
- Include the **primary keys** of both participating entities.
  - These form the **composite primary key** of the relation.
- Add any **attributes of the relationship**.

---

### **Step 6: Mapping of Multivalued Attributes**

- Create a **new relation**.
- Include:
  - The **multivalued attribute**.
  - The **primary key** of the entity to which it belongs.
- The **primary key** is the **combination** of these two attributes.

---

### **Step 7: Mapping of N-ary Relationship Types (n > 2)**

- Create a **new relation**.
- Include the **primary keys of all participating entities**.
  - These together form the **composite primary key**.
- Include any **attributes of the relationship**.


In [1]:
# ----------------------------------------
# Step 0: Define basic classes to model ER constructs
# ----------------------------------------

class Attribute:
    def __init__(self, name, is_key=False, is_multivalued=False):
        self.name = name
        self.is_key = is_key
        self.is_multivalued = is_multivalued

    def __repr__(self):
        flags = []
        if self.is_key: flags.append("PK")
        if self.is_multivalued: flags.append("MV")
        return f"{self.name}{'(' + ', '.join(flags) + ')' if flags else ''}"


class EntityType:
    def __init__(self, name, is_weak=False, owner=None):
        self.name = name
        self.is_weak = is_weak
        self.owner = owner  # Owner EntityType (for weak entities)
        self.attributes = []

    def add_attribute(self, attr):
        self.attributes.append(attr)

    def __repr__(self):
        return f"{self.name}({', '.join(map(str, self.attributes))})"


class Relationship:
    def __init__(self, name, entities, cardinality, attributes=None, total_participation=None):
        self.name = name
        self.entities = entities  # List of participating entities
        self.cardinality = cardinality  # e.g., ('1', 'N') or ('M', 'N')
        self.attributes = attributes or []
        self.total_participation = total_participation or [False] * len(entities)

    def __repr__(self):
        parts = [f"{e.name}:{c}" for e, c in zip(self.entities, self.cardinality)]
        return f"{self.name}({', '.join(parts)})"


# ----------------------------------------
# Step 1–7: Mapping Function
# ----------------------------------------

def map_er_to_relational(entities, relationships):
    relations = []

    # Step 1: Regular (Strong) Entities
    for e in entities:
        if not e.is_weak:
            attrs = [a.name for a in e.attributes if not a.is_multivalued]
            relation = {
                "name": e.name,
                "attributes": attrs,
                "primary_key": [a.name for a in e.attributes if a.is_key]
            }
            relations.append(relation)

    # Step 2: Weak Entities
    for e in entities:
        if e.is_weak and e.owner:
            attrs = [a.name for a in e.attributes]
            owner_key = [a.name for a in e.owner.attributes if a.is_key]
            relation = {
                "name": e.name,
                "attributes": owner_key + attrs,
                "primary_key": owner_key + [a.name for a in e.attributes if a.is_key]
            }
            relations.append(relation)

    # Step 3–5: Relationships
    for r in relationships:
        if len(r.entities) == 2:
            e1, e2 = r.entities
            c1, c2 = r.cardinality

            # Step 3: 1:1
            if c1 == '1' and c2 == '1':
                fk_entity = e1 if r.total_participation[0] else e2
                owner = e2 if fk_entity == e1 else e1
                for rel in relations:
                    if rel["name"] == fk_entity.name:
                        rel["attributes"].append(owner.attributes[0].name)
                        if True in r.total_participation:
                            rel["not_null"] = owner.attributes[0].name
                        rel["attributes"] += [a.name for a in r.attributes]
            # Step 4: 1:N
            elif ('1' in r.cardinality and 'N' in r.cardinality) or ('1' in r.cardinality and 'M' in r.cardinality):
                n_side_index = r.cardinality.index('N') if 'N' in r.cardinality else r.cardinality.index('M')
                n_entity = r.entities[n_side_index]
                one_entity = r.entities[1 - n_side_index]
                for rel in relations:
                    if rel["name"] == n_entity.name:
                        rel["attributes"].append(one_entity.attributes[0].name)
                        rel["attributes"] += [a.name for a in r.attributes]
            # Step 5: M:N
            elif c1 in ['M', 'N'] and c2 in ['M', 'N']:
                relation = {
                    "name": r.name,
                    "attributes": [a.attributes[0].name for a in r.entities] + [a.name for a in r.attributes],
                    "primary_key": [a.attributes[0].name for a in r.entities]
                }
                relations.append(relation)

        # Step 7: N-ary Relationships
        elif len(r.entities) > 2:
            relation = {
                "name": r.name,
                "attributes": [e.attributes[0].name for e in r.entities] + [a.name for a in r.attributes],
                "primary_key": [e.attributes[0].name for e in r.entities]
            }
            relations.append(relation)

    # Step 6: Multivalued Attributes
    for e in entities:
        for a in e.attributes:
            if a.is_multivalued:
                relation = {
                    "name": f"{e.name}_{a.name}",
                    "attributes": [attr.name for attr in e.attributes if attr.is_key] + [a.name],
                    "primary_key": [attr.name for attr in e.attributes if attr.is_key] + [a.name]
                }
                relations.append(relation)

    return relations


# ----------------------------------------
#  Sample Test Case
# ----------------------------------------

# Entities
student = EntityType("Student")
student.add_attribute(Attribute("SID", is_key=True))
student.add_attribute(Attribute("Name"))
student.add_attribute(Attribute("Hobbies", is_multivalued=True))

course = EntityType("Course")
course.add_attribute(Attribute("CID", is_key=True))
course.add_attribute(Attribute("Title"))

enroll = Relationship("Enrolls", [student, course], cardinality=('M', 'N'), attributes=[Attribute("Grade")])

# Run mapping
relational_schema = map_er_to_relational([student, course], [enroll])

# Display
print(" Relational Schema Output:")
for r in relational_schema:
    print(f"Table: {r['name']}")
    print(f" - Attributes: {r['attributes']}")
    print(f" - Primary Key: {r['primary_key']}")
    print()


 Relational Schema Output:
Table: Student
 - Attributes: ['SID', 'Name']
 - Primary Key: ['SID']

Table: Course
 - Attributes: ['CID', 'Title']
 - Primary Key: ['CID']

Table: Enrolls
 - Attributes: ['SID', 'CID', 'Grade']
 - Primary Key: ['SID', 'CID']

Table: Student_Hobbies
 - Attributes: ['SID', 'Hobbies']
 - Primary Key: ['SID', 'Hobbies']



## 9.2 Mapping EER Model Constructs to Relations

This section extends the ER-to-relational mapping algorithm to include EER constructs such as **specialization**, **generalization**, **shared subclasses**, and **categories**.

---

### **9.2.1 Mapping of Specialization or Generalization**

#### **Step 8: Options for Mapping Specialization or Generalization**

Given a superclass **C** with key **k** and attributes $\{a_1, ..., a_n\}$, and **m subclasses** $\{S_1, S_2, ..., S_m\}$, choose one of the following options:

---

#### **Option 8A: Multiple relations — superclass and subclasses**

- Create relation **L** for **C** with attributes $\{k, a_1, ..., a_n\}$  
  - **PK(L) = k**
- Create relation **Lᵢ** for each subclass **Sᵢ** with attributes $\{k\} \cup \{\text{attributes of } Sᵢ\}$  
  - **PK(Lᵢ) = k**
- **Supports**: Any specialization (disjoint/overlapping, total/partial)

---

#### **Option 8B: Multiple relations — subclass relations only**

- Create relation **Lᵢ** for each subclass **Sᵢ** with attributes $\{k, a_1, ..., a_n\} \cup \{\text{attributes of } Sᵢ\}$  
  - **PK(Lᵢ) = k**
- **Supports**: Total, disjoint specialization only
- **Drawbacks**:
  - Redundancy for overlapping subclasses
  - Loss of superclass entities if not total

---

#### **Option 8C: Single relation with one type attribute**

- Create one relation **L** with attributes:  
  $\{k, a_1, ..., a_n\} \cup \{\text{attributes of } S_1\} \cup ... \cup \{\text{attributes of } S_m\} \cup \{t\}$  
  - **PK(L) = k**
- **t** is a type (discriminating) attribute indicating subclass membership
- **Supports**: Disjoint specialization only
- **Drawback**: Many NULLs for subclass-specific attributes

---

#### **Option 8D: Single relation with multiple type attributes**

- Create one relation **L** with attributes:  
  $\{k, a_1, ..., a_n\} \cup \{\text{attributes of } S_1\} \cup ... \cup \{\text{attributes of } S_m\} \cup \{t_1, t_2, ..., t_m\}$  
  - **PK(L) = k**
- Each **tᵢ** is a Boolean flag indicating membership in subclass **Sᵢ**
- **Supports**: Overlapping specializations (also works for disjoint)
- **Drawback**: Still may result in NULLs

---

**Multilevel Specialization Notes:**

- Apply different options at different levels in a hierarchy/lattice
- **Example**:
  - Use **8A** for `PERSON → {EMPLOYEE, STUDENT}`
  - Use **8D** for `STUDENT_ASSISTANT → {TA, RA}`

---

### **9.2.2 Mapping of Shared Subclasses (Multiple Inheritance)**

- A **shared subclass** (e.g., `ENGINEERING_MANAGER`) inherits from multiple superclasses
- **Requirement**: Superclasses must have the **same key**
- Apply **any Step 8 option**, respecting specialization constraints
- **Example**:  
  - `STUDENT_ASSISTANT`:
    - Mapped using **Option 8C** in `EMPLOYEE` (type attribute)
    - Mapped using **Option 8D** in `STUDENT` (flag attribute)

---

### **9.2.3 Mapping of Categories (Union Types)**

#### **Step 9: Mapping of Union Types (Categories)**

- Used when a **subclass (category)** is a **subset of multiple superclasses with different keys**
- **Example**: `OWNER` is a category of `PERSON`, `BANK`, and `COMPANY`

---

**Procedure:**

- Create a new relation for the category (e.g., `OWNER`)
- Introduce a **surrogate key** (e.g., `Owner_id`) for the category
- Include **attributes of the category**
- Include `Owner_id` as a **foreign key** in each superclass relation
- **Optional**: Add a **type attribute** to indicate source superclass

---

**If Superclasses Have the Same Key**:

- **Example**: `REGISTERED_VEHICLE` from `CAR` and `TRUCK`
- No surrogate key needed
- One relation can directly map the category using the shared key (e.g., `Vehicle_id`)


In [2]:
# Reuse base classes from before: Attribute, EntityType, Relationship

class Specialization:
    def __init__(self, superclass, subclasses, option="8A", disjoint=True, total=False):
        self.superclass = superclass
        self.subclasses = subclasses
        self.option = option
        self.disjoint = disjoint
        self.total = total

class Category:
    def __init__(self, name, superclasses, attributes, use_surrogate=True):
        self.name = name
        self.superclasses = superclasses
        self.attributes = attributes
        self.use_surrogate = use_surrogate


def map_eer_constructs(entities, specializations=[], categories=[]):
    relations = []

    # Handle entities first (same as Step 1)
    for e in entities:
        attrs = [a.name for a in e.attributes if not a.is_multivalued]
        relation = {
            "name": e.name,
            "attributes": attrs,
            "primary_key": [a.name for a in e.attributes if a.is_key]
        }
        relations.append(relation)

    # ---- Step 8: Specialization/Generalization ----
    for spec in specializations:
        C = spec.superclass
        subs = spec.subclasses
        option = spec.option.upper()

        if option == "8A":
            # One relation for superclass + one per subclass
            relations.append({
                "name": C.name,
                "attributes": [a.name for a in C.attributes],
                "primary_key": [a.name for a in C.attributes if a.is_key]
            })
            for S in subs:
                attrs = [a.name for a in S.attributes]
                pk = [a.name for a in C.attributes if a.is_key]
                relations.append({
                    "name": S.name,
                    "attributes": pk + attrs,
                    "primary_key": pk
                })

        elif option == "8B":
            # Subclass-only relations with inherited + subclass attributes
            for S in subs:
                attrs = [a.name for a in C.attributes] + [a.name for a in S.attributes]
                pk = [a.name for a in C.attributes if a.is_key]
                relations.append({
                    "name": S.name,
                    "attributes": attrs,
                    "primary_key": pk
                })

        elif option == "8C":
            # Single relation with type discriminator
            type_attr = "type"
            all_attrs = list({a.name for a in C.attributes})
            for S in subs:
                all_attrs += [a.name for a in S.attributes]
            relations.append({
                "name": C.name + "_AllInOne",
                "attributes": list(set(all_attrs + [type_attr])),
                "primary_key": [a.name for a in C.attributes if a.is_key]
            })

        elif option == "8D":
            # Single relation with boolean flags per subclass
            flags = [f"is_{S.name}" for S in subs]
            all_attrs = list({a.name for a in C.attributes})
            for S in subs:
                all_attrs += [a.name for a in S.attributes]
            relations.append({
                "name": C.name + "_AllInOneFlags",
                "attributes": list(set(all_attrs + flags)),
                "primary_key": [a.name for a in C.attributes if a.is_key]
            })

    # ---- Step 9: Categories (Union Types) ----
    for cat in categories:
        if cat.use_surrogate:
            surrogate = cat.name + "_id"
            relation = {
                "name": cat.name,
                "attributes": [surrogate] + [a.name for a in cat.attributes],
                "primary_key": [surrogate],
                "foreign_keys": [(superclass.name, surrogate) for superclass in cat.superclasses]
            }
        else:
            # Shared key case
            key_name = cat.superclasses[0].attributes[0].name  # Assume same key
            relation = {
                "name": cat.name,
                "attributes": [key_name] + [a.name for a in cat.attributes],
                "primary_key": [key_name]
            }
        relations.append(relation)

    return relations


In [3]:
# Define Superclass and Subclasses
person = EntityType("Person")
person.add_attribute(Attribute("PID", is_key=True))
person.add_attribute(Attribute("Name"))

employee = EntityType("Employee")
employee.add_attribute(Attribute("Job"))

student = EntityType("Student")
student.add_attribute(Attribute("Major"))

# Specialization: Person → Employee, Student (Disjoint, Total)
spec1 = Specialization(superclass=person, subclasses=[employee, student], option="8A", disjoint=True, total=True)

# Category: OWNER ← {Person, Company}
company = EntityType("Company")
company.add_attribute(Attribute("CID", is_key=True))
company.add_attribute(Attribute("CName"))

owner_attrs = [Attribute("Ownership_Date")]
category1 = Category(name="Owner", superclasses=[person, company], attributes=owner_attrs, use_surrogate=True)

# Run mapping
eerschema = map_eer_constructs([person, employee, student, company], specializations=[spec1], categories=[category1])

# Display
print(" EER-to-Relational Mapping Output:")
for r in eerschema:
    print(f"Table: {r['name']}")
    print(f" - Attributes: {r['attributes']}")
    print(f" - Primary Key: {r['primary_key']}")
    if 'foreign_keys' in r:
        print(f" - Foreign Keys: {r['foreign_keys']}")
    print()


📘 EER-to-Relational Mapping Output:
Table: Person
 - Attributes: ['PID', 'Name']
 - Primary Key: ['PID']

Table: Employee
 - Attributes: ['Job']
 - Primary Key: []

Table: Student
 - Attributes: ['Major']
 - Primary Key: []

Table: Company
 - Attributes: ['CID', 'CName']
 - Primary Key: ['CID']

Table: Person
 - Attributes: ['PID', 'Name']
 - Primary Key: ['PID']

Table: Employee
 - Attributes: ['PID', 'Job']
 - Primary Key: ['PID']

Table: Student
 - Attributes: ['PID', 'Major']
 - Primary Key: ['PID']

Table: Owner
 - Attributes: ['Owner_id', 'Ownership_Date']
 - Primary Key: ['Owner_id']
 - Foreign Keys: [('Person', 'Owner_id'), ('Company', 'Owner_id')]



## 9.3 Summary

### **Section 9.1:**
- Introduced the **ER-to-relational mapping algorithm**, converting **ER model elements** into a **relational schema**.
- Used examples from the **COMPANY database**.
- **Table 9.1** summarized the mapping between **ER constructs** and their **relational counterparts**.

---

### **Section 9.2:**
- Extended the algorithm to handle **EER model constructs**, including:
  - **Specialization / Generalization**
  - **Shared Subclasses** (Multiple Inheritance)
  - **Categories** (Union Types)

---

### **Relevance:**
- These mapping steps are **foundational in automated schema generation**.
- Often embedded in **graphical database design tools**.
- Enable efficient conversion from **conceptual models** to **relational schemas**.
